# Robot Harness Optimization

[RHO](https://arxiv.org/abs/2606.16458) (*Your Coding Agent is Secretly a Roboticist*) optimizes robot policy **source code** before deployment, so the deployed system runs frozen Python with no language model in the control loop. It instantiates this with **HELIX**, which treats a whole multi-file repository as the evolving candidate and uses a tool-enabled coding agent as the mutation operator.

The search repeats five steps:

1. Evaluate the seed repository on the validation instances.
2. Sample a parent from the frontier and let a coding agent edit the repository.
3. Gate the child on a training minibatch, rejecting anything that regresses.
4. Evaluate the survivors on validation and update the frontier.
5. Deploy the highest mean-reward member of the final frontier.

Notebook 3 asked a local model to write a fresh policy for every rollout and measured how much the results varied. RHO moves the model upstream of deployment, so what ships is reviewable code rather than a weight delta or a prompt.

## Goals

* Read an evolution experiment as a configuration: the objective, the editable surface, the protected evaluator, and the acceptance gate
* Read a recorded run across two robot tasks and six validation layouts, and work out how much of it was useful
* Watch the seed and the deployed policy run the same layout live, and read the gap between them
* Say what those scores support and what they do not, given that the search optimized against the same layouts

## What this notebook reads

Most of what follows replays an evolution run that was recorded ahead of time.
The cell below reports which of its pieces are on disk: the study report, the
seed repository, the repository the search deployed, and the configuration
files that defined the run.

In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

sys.path.insert(0, "/ryzers/notebooks/scripts")

# capx_demo chdir()s into the CaP-X tree on import, so every path in this
# notebook stays absolute. None of these imports touch the GPU.
import capx_demo
import rho_demo
import rho_multitask_demo
import rho_report

RECORDED_ROOT = Path("/ryzers/notebooks/recorded_results")
SEED_REPO = RECORDED_ROOT / "repos" / "seed"
BEST_REPO = RECORDED_ROOT / "repos" / "selected"

# Keep live rollout videos out of the read-only recorded tree.
rho_demo.VIDEO_ROOT = Path("/tmp/rho_live_videos")

STATUS = rho_report.preflight(RECORDED_ROOT)
print(rho_report.format_preflight(STATUS))

## What the search actually edits

The evolving candidate is a whole repository, divided into an editable policy surface and a protected evaluation boundary.

Two task policies sit on the editable surface. Each one is a real program that a local Gemma E4B model produced during a CaP-X run, kept with its original failure intact so that the mutator has something genuine to diagnose:

| Policy | Task | How the seed fails |
| --- | --- | --- |
| `solver/tasks/cube_stack.py` | stack a red cube on a green cube | indexes a flat XYZ pose as if it were nested, raising `IndexError` after the grasp |
| `solver/tasks/cube_lift.py` | pick up the red cube and lift it clear of the table | calls `numpy.array` without importing numpy, raising `NameError` |

Each failure is drawn from a real sweep rather than written by hand. The missing-import failure, for instance, appeared in 9 of 40 cube-lift trials. `provenance.json` records the source trial and a checksum of the original program for every seed.

Both seeds fail on every layout they were scanned against. A seed that already succeeds leaves the search no room to improve on that task, and makes a real repair hard to tell apart from a lucky rollout. A third policy, `spill_wipe`, was dropped from this experiment for that reason, since its seed already solved six of the eight layouts it was scanned on.

`solver/geometry.py` and `solver/runtime.py` start out as empty modules. The mutator may add shared helpers there, and either policy may import them. Stacking and lifting both need grasp and pose geometry, so there is real shared structure to factor out, which is what makes this a multi-file search rather than two independent single-file repairs.

In [ ]:
rho_report.require_ready(STATUS)

import tomllib

SEED_CONFIG = tomllib.loads((SEED_REPO / "helix.toml").read_text())
PROTECTED = set(SEED_CONFIG["evaluator"]["protected_files"]) | {"helix.toml"}

surface = []
for path in sorted(SEED_REPO.rglob("*")):
    if not path.is_file() or "__pycache__" in path.parts:
        continue
    relative = path.relative_to(SEED_REPO).as_posix()
    if relative in PROTECTED:
        role = "protected — mutations that touch it are rejected"
    elif relative.startswith("solver/"):
        role = "EDITABLE — the search space"
    else:
        role = "reference"
    surface.append(
        {
            "file": relative,
            "role": role,
            "lines": len(path.read_text().splitlines()),
        }
    )

display(pd.DataFrame(surface).set_index("file"))

## The experiment definition

`helix.toml` is the entire experiment, printed below without filtering. Most of the design work in an evolution run goes into two string fields rather than into the numeric settings: `objective` states what a good repository looks like, and `agent.background` tells the mutator how to behave inside the loop. Those two fields move the outcome more than any number in the file.

Among the numeric settings, two change the character of the search more than the rest. `acceptance_criterion = "strict_improvement"` decides which children survive their parent, since a child that regresses on its training minibatch is discarded before it ever reaches validation. `frontier_type = "instance"` decides what the population keeps, retaining the best candidate for each validation layout rather than collapsing onto a single best-on-average one. Nearly everything else is budget: how many generations to run, how many proposals to draw per generation, and a hard cap on evaluations that applies regardless.

In [ ]:
print((SEED_REPO / "helix.toml").read_text())

## Running a full evolution

`agent.model` in that file names a 30B coding model, and the size matters in a course about running small models locally. Asked directly for the repair a seed policy needs, the smaller local models write it correctly. They come apart inside the agent loop, where the model has to locate the bug, choose an edit, and sequence tool calls over a long context. Gemma E4B produced no accepted mutation across eight generations of trying. What separates the two models here is driving the loop rather than writing the Python.

The cell below prints the command that produced the recorded study used later in this notebook. It is shown rather than run, because it takes about an hour.

Two of its flags need explaining. `--generations 2` is short on purpose: a longer version of this same experiment was run first, and everything it achieved happened in the first generation, so the recorded study was cut to the length the search needed. `--hidden-repeats` asks the study to also replay layouts the search never sampled. That evidence is recorded but not shown here, because the sections below stay on the six validation layouts to keep the session short.

In [ ]:
print("===== the command that produced the recorded study =====")
print(rho_report.REGENERATE_COMMAND)

## The evaluator boundary

An agent that can edit its own scorer will eventually optimize the scorer instead of the policy. The RHO authors report observing exactly that from mutators given shell access, so the mutator and the evaluator are separated by three mechanisms.

1. **Protected files are hashed.** At run start HELIX writes SHA-256 digests of
   `probe.py`, `helix.toml`, `opencode.json`, `CONTRACT.md`, `scenarios.json`
   and `provenance.json` into `.helix/evaluator_manifest.json`. A candidate that
   modifies any of them is rejected *before* it is scored, so editing the
   benchmark is not a viable strategy.
2. **Tool permissions are allow-listed.** `opencode.json` denies edits outside
   `solver/`, denies web fetch and search, denies sub-agent and skill tools, and
   permits exactly two shell commands: a compile check and the self-check probe.
3. **Scores come back over a fixed protocol.** `probe.py` prints one
   `HELIX_RESULT=[[score, side_info], ...]` line, positionally matched to the
   example ids HELIX wrote into `helix_batch.json`. The `side_info` payload
   carries reward, completion, traceback and evaluator feedback, and that is
   what becomes the reflective prompt for the next mutation.

The mutator never sees the raw `HELIX_RESULT` line, only the diagnostics.

In [ ]:
print("===== the evaluator the candidate repository runs =====")
print((SEED_REPO / "probe.py").read_text())

print("===== what the mutator is allowed to do =====")
permissions = json.loads((SEED_REPO / "opencode.json").read_text())["permission"]
print(json.dumps({key: permissions[key] for key in ("edit", "bash")}, indent=2))

## One generation, live

The command above runs for hours. A single generation is small enough to watch, and it shows what the recorded run below is made of: the coding agent reads the evaluator's diagnostics, edits the repository, and the gate scores the result against the parent.

Set `RUN_LIVE = True` to run one generation on a fresh copy of the seed repository. It starts the mutation model and the robot services first, then takes several minutes, and it can end with nothing accepted. A rejected generation is a normal outcome, and the run recorded below rejected four of them. Leave the flag off and the cell prints what would happen instead.

In [ ]:
RUN_LIVE = False

LIVE_ROOT = Path("/tmp/rho_live_generation/candidate")

if RUN_LIVE:
    # A live generation needs both the mutation model and the robot services,
    # which the sections below start on their own.
    rho_demo.ensure_services(model=rho_demo.DEFAULT_RHO_MODEL)

    started = perf_counter()
    # One proposal instead of four keeps a single generation watchable; the
    # minibatch still covers every task, so the gate sees the whole repository.
    live_repo = rho_multitask_demo.prepare_workshop(
        LIVE_ROOT,
        generations=2,
        helix_overrides={"num_parallel_proposals": 1},
    )
    run = rho_multitask_demo.run_helix(live_repo, generations=1, timeout_seconds=1200)
    frontier = rho_multitask_demo.frontier_summary(live_repo)

    children = {
        candidate_id: candidate
        for candidate_id, candidate in frontier["candidates"].items()
        if candidate_id != "g0-s0"
    }
    print(f"\nelapsed {perf_counter() - started:.0f}s · timed out: {run.timed_out}")

    if children:
        best = max(
            children,
            key=lambda c: sum(children[c].get("scores", {}).values()),
        )
        display(
            pd.DataFrame(
                [
                    {"candidate": cid, **candidate.get("scores", {})}
                    for cid, candidate in sorted(frontier["candidates"].items())
                ]
            ).set_index("candidate")
        )
        print(f"\n===== what {best} changed =====")
        print(
            rho_demo.source_diff(live_repo, live_repo / ".helix" / "worktrees" / best)
            or "(no source change)"
        )
    else:
        print("\nNothing cleared the training gate this generation.")
        print("That is a normal outcome; the recorded run below rejected several.")
else:
    print("RUN_LIVE = False")
    print()
    print("Set RUN_LIVE = True to prepare a fresh seed repository and run one")
    print("generation with a single proposal. Expect several minutes, and expect")
    print("that it may not produce an accepted candidate.")

---

## A recorded full run

Everything from here on reads a study recorded ahead of time: the same two tasks, the same seed repository, and the same mutator, run for two generations with four proposals each. The cell below loads it and reports what it cost.

In [ ]:
REPORT = rho_report.load_report(RECORDED_ROOT)
COST = rho_report.study_cost(REPORT)
TASKS = rho_report.validation_tasks(REPORT)
DEPLOYED = REPORT["selected_candidate"]

print(f"Tasks:            {', '.join(TASKS)}")
print(f"Mutation model:   {COST['mutation_model']}")
print(f"Generations:      {COST['generations']}"
      f" · {COST['proposal_slots_per_generation']} proposal slots each")
print(f"Deployed:         {DEPLOYED}")
print(f"Evolution time:   {COST['evolution_seconds'] / 60:.1f} min")
print(f"Total study time: {COST['total_seconds'] / 60:.1f} min")
print()
print(rho_report.selection_rule(REPORT))

## Search progress

Selection deploys the candidate with the highest mean validation reward, so plotting that mean against generation shows what the search was actually buying. Every proposal that cleared the training gate appears as a dot, the line tracks the best score reached so far, and the crosses along the bottom count the proposals the gate rejected in each generation.

Expect the line to jump on the first generation and then flatten. With a 30B mutator reading a real traceback, an `IndexError` from a mis-indexed pose and a `NameError` from a missing import are close to mechanical repairs, and the search finds them immediately.

An earlier six-generation version of this experiment reached a perfect validation score in generation 1. Generations 2 through 6 then rejected 16 of their 20 proposals, and the one candidate that survived to be deployed differed from the generation-1 winner only by a `try/except` that re-raises. Replayed over 30 rollouts, the two scored identically. Search budget past the point of solving the problem buys churn, and measuring is the only way to tell which one you bought.

The table underneath gives the run as lineage: which parent each candidate came from, which files it touched, what the gate decided, and the mean validation reward it reached. The seed sits at the top on 0.000, since it crashes on all six layouts, and the proposals the gate rejected have no score at all because they were discarded before validation ever ran.

Those six layouts are the ones the search optimized against, so read a 1.000 as evidence that the repair works rather than as a measure of how far it generalizes.

In [ ]:
rho_report.plot_progress(REPORT)

PROGRESS = rho_report.progress_frame(REPORT)
print(f"Proposals made:     {len(PROGRESS) - 1}")
print(f"Cleared the gate:   {int(PROGRESS['validated'].sum()) - 1}")
print(f"Rejected:           {int((~PROGRESS['validated']).sum())}")
print()
display(rho_report.lineage_frame(REPORT))

## What the deployed repository changed

The diff below compares the deployed candidate against the seed, split by file. This is the artifact RHO ships. A reviewer can read it line by line and accept or reject it, which a weight delta does not allow. Nothing in it calls a language model, so the deployed system runs the same way every time.

In [ ]:
display(rho_report.diff_summary(REPORT["selected_diff"]))

for name, section in rho_report.diff_sections(REPORT["selected_diff"]).items():
    print(f"\n===== {name} =====")
    print(section)

## Start the robot services

The replay below runs in simulation, which needs the perception and control
stack loaded first: OWLv2 for text-conditioned box grounding, SAM2 for masking,
Contact-GraspNet for grasp proposals, and PyRoKi for IK.

None of that is a language model. The deployed policy is frozen Python and
nothing in its control loop calls a model. Lemonade loads here because the
optional live generation above needs a mutation agent.


In [ ]:
SETUP_STARTED = perf_counter()
rho_demo.ensure_services(model=rho_demo.DEFAULT_RHO_MODEL)
print(f"Services ready in {perf_counter() - SETUP_STARTED:.1f}s")


## Watch both policies run

Everything above is a recording. The cell below runs the two frozen repositories live, one validation layout per task, so you can watch the seed fail and the deployed policy finish. Four rollouts, ten to sixteen seconds each.

Watch the two policies for different things. The seed crashing is certain, because the bug is in the code rather than in the physics, and it raised the same exception in all 30 replays behind these numbers. The deployed policy completing is likely and not certain: across those same 30 replays it solved 26. The two layouts below are the ones that came out 5 for 5, and grasp sampling decides the rest.

In [ ]:
print("Both runs execute frozen Python: no generation, no tool loop, no LLM.")

for task in TASKS:
    trial = rho_report.validation_trial(REPORT, task)
    live_rollouts = []
    for label, repo in (("seed", SEED_REPO), (f"deployed ({DEPLOYED})", BEST_REPO)):
        started = perf_counter()
        result = rho_multitask_demo.replay_trials(
            repo,
            trials={task: [trial]},
            capture=True,
        )[0]
        result["wall_seconds"] = perf_counter() - started
        live_rollouts.append((label, result))

    print()
    print(f"===== {task.replace('_', ' ')} · validation trial {trial} =====")
    display(
        pd.DataFrame(
            [
                {
                    "policy": label,
                    "reward": result["reward"],
                    "raw reward": result.get("raw_reward"),
                    "solved": result["task_completed"],
                    "seconds": round(result["wall_seconds"], 1),
                }
                for label, result in live_rollouts
            ]
        ).set_index("policy")
    )
    capx_demo.show_comparison_grid(live_rollouts)

## Taking the deployed repository with you

The deployed candidate is a directory of Python on this machine, so it outlives the notebook. The cell below prints where it is, what it contains, and the two lines that replay any layout against it, which is also how you would extend the comparison to layouts the search never saw.

In [ ]:
print(f"Deployed repository: {BEST_REPO}")
for path in sorted(BEST_REPO.rglob("*.py")):
    print(f"  {path.relative_to(BEST_REPO)}")

print()
print("===== replay it on any layout =====")
print(f"""import sys; sys.path.insert(0, "/ryzers/notebooks/scripts")
import rho_multitask_demo

rho_multitask_demo.replay_trials(
    "{BEST_REPO}",
    trials={{"cube_stack": [11, 12, 13]}},   # any trial ids you like
)""")

---

## Key takeaways

* **The candidate is a whole repository.** RHO evolves several source files with imports between them, so a mutation can add a shared helper in one file and call it from another. What ships is code a reviewer can read.
* **Deployment runs no model.** The replays above execute frozen Python against the robot services. The language model cost is paid up front, during the search, rather than on every rollout as in notebook 3.
* **The training gate does most of the filtering.** A child must strictly improve on its parent's minibatch before it is validated at all. Most proposals fail that test, and those rejections are where the bulk of the compute goes.
* **The mutator has to drive a tool loop.** Small local models wrote correct repairs when asked directly and still produced no accepted mutation inside the agent loop.
* **Match the search budget to the problem, then measure it more than once.** These bugs fell to the first generation, and a longer run of the same experiment spent five more generations producing a candidate that measured identically. Replayed five times each, these six layouts came out 26 for 30, and the claims that held still were the deterministic ones, like whether the code raises.

## What to try next

* Replay the deployed repository on trial ids the search never used, with the snippet printed above, and see how much of the validation result survives. Run each layout several times, since single rollouts move enough that one pass will not settle it. `scripts/rho_replay_scan.py` does this for the validation layouts and prints a pass rate per layout.
* Add a third task by dropping a policy into `solver/tasks/` and adding it to the `TASKS` registry, then check whether the mutator starts factoring shared code into `solver/geometry.py`. Scan the new seed across several trials first: a seed that already succeeds gives the search nothing to work with.
* Set `RUN_LIVE = True`, then read the agent's diff alongside the evaluator feedback it was given and judge whether the edit follows from the diagnostics.

## References

* [RHO: Your Coding Agent is Secretly a Roboticist](https://arxiv.org/abs/2606.16458) — the paper this notebook follows
* [Code as Policies](https://code-as-policies.github.io/) — the policy-as-source-code framing the seed programs build on
* `scripts/rho_multitask_demo.py` — the experiment definition used here
* `scripts/rho_report.py` — the reporting and plotting helpers

---

## Teardown

The cell below stops the model and robotics services this notebook started. Run it last, since the comparisons above need them.

In [ ]:
rho_demo.stop_owned_services()
print("Notebook-owned model and robotics services stopped.")